In [1]:
import pandas as pd     #   library pandas

In [ ]:
data = pd.read_csv('File Path')        #   read the csv data after it has been cleaned and validated

Remove columns that contain PII or are unnecessary

In [16]:
data = data.drop(['originaladdress1',      #   personally identifiable information
           'originaladdress2',      #   personally identifiable information
           'originalcity',      #   replacing with validated city data
           'originalzip',       #   replacing with validated zip data (will privatize in the next step)
           'pin',       #   possibly personally identifiable information
           'contractortrademapped',     #   'contractortrade' column has more information (this columns is effectively the same as 'contractortrade' but without information about unlicenced contractors)
           'contractorphone',      #   personally identifiable information
           'contractoraddress1',      #   personally identifiable information
           'contractoraddress2',      #   personally identifiable information
           'ownername',      #   personally identifiable information
           'owneraddress1',      #   personally identifiable information
           'owneraddress2',      #   personally identifiable information
           'latitude',      #   personally identifiable information
           'longitude']      #   personally identifiable information
           , axis = 1)

Edit columns to remove PII

In [17]:
data['index1'] = data.index     #   creates a column 'index1' that saves the index of the original order of the data
data = data.astype({'index1': str})     #   saves the 'index1' column's datatype as a string
data['index1'] = [x.zfill(4) for x in data['index1']]       #   fills each entry with zeros until the string is 4 characters long

In [18]:
data = data.sort_values(by = 'applieddate')     #   sorts the dataset by the 'applieddate' column in ascending order
data.reset_index(drop = True, inplace = True)       #   resets the index for the newly sorted dataset (earliest permits are first)

In [19]:
data['index2'] = data.index     #   creates a column 'index2' that saves the index of the new order of the data
data = data.astype({'index2': str})     #   saves the 'index2' column's datatype as a string
data['index2'] = [x.zfill(4) for x in data['index2']]       #   fills each entry with zeros until the string is 4 characters long

In [ ]:
data['permitnum'] = [z[2:5] + y + x for x, y, z in zip(data['index1'], data['index2'], data['applieddate'])]        #   replaces the potential personally identifiable information from the 'permitnum' column with a new permit number that is the last two digits of the year followed by sorted index (filled with zeroes up to 4 characters) then the original index (filled with zeroes up to 4 characters). This best matches the format of the original 'permitnum' column

In [21]:
data = data.drop(['index1','index2'], axis = 1)     #   drops the 'index1' and 'index2' columns after they've been used to generate a new permit number

In [22]:
data['contractorzip'] = [x[:5] for x in data['contractorzip'].fillna('')]       #   makes all zip codes the first 5 digits and fills an NA data with '' for indexing purposes
data['ownerzip'] = [x[:5] for x in data['ownerzip'].fillna('')]       #   makes all zip codes the first 5 digits and fills an NA data with '' for indexing purposes

In [23]:
data['originalcity_validated'] = [x.split(', ')[1] for x in data['validated_address']]      #   extracts the city data from the 'validated_address' column and saves it to a new column 'originalcity_validated'
data['originalzip_validated'] = [x.split(', ')[2][3:8] for x in data['validated_address']]      #   extracts the first 5 digits of the zipcode from the 'validated_address' column and saves it to a new column ' originalzip_validated'

In [24]:
data = data.drop('validated_address', axis = 1)     #   removes the 'validated_address' column since it contains personally identifiable information